In [17]:
!pip install PyPortfolioOpt
!pip install scikit-learn
!pip install stable-baselines3
!pip install gymnasium
!pip install pypfopt
!pip install tqdm
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.1/220.1 kB 8.2 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement pypfopt (from versions: none)
ERROR: No matching distribution found for pypfopt


In [26]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime
from tqdm import tqdm
from stable_baselines3.common.noise import NormalActionNoise

### **CREATING AN INTERATOR FOR BACTH PROCESSING**

In [27]:
class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 100, train_years: int = 20):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 9  # Updated with new features (BB_upper, BB_lower)

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):
        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        # Process one ticker at a time to avoid memory issues
        for tic in tqdm(self.tickers, desc="Processing tickers"):
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Bollinger Bands
            stock_data['BB_upper'] = stock_data['Price_MA'] + (stock_data['Volatility'] * 2)
            stock_data['BB_lower'] = stock_data['Price_MA'] - (stock_data['Volatility'] * 2)

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        for tic in tqdm(self.tickers, desc="Scaling features"):
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(tqdm(self.unique_dates, desc="Processing dates")):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self

### **ENVIRONMENT CREATION FOR TRAIN SAC MODEL**

In [28]:
import gym
import numpy as np
from gym import spaces

class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features
        self.episode_reward = 0
        self.episode_steps = 0
        self.debug_info = {}

        # Adjust action space for SAC: outputs are in [-1,1], so we will scale them to [0,1]
        self.action_space = spaces.Box(
            low=-1, high=1, shape=(self.num_stocks + 1,), dtype=np.float32
        )

        # Normalized observation space
        self.observation_space = spaces.Box(
            low=-1, high=1, shape=(self.num_stocks, self.lookback, self.num_features), dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)
        self.episode_reward = 0
        self.episode_steps = 0
        self.debug_info = {}

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        obs = self.current_step['features'].astype(np.float32)
        return np.clip(obs, -1, 1)  # Normalized observation

    def step(self, action):
        self.episode_steps += 1

        # Scale SAC action output from [-1, 1] to [0, 1]
        action = (action + 1) / 2
        action = np.clip(action, 0, 1)

        if action.shape != (self.num_stocks + 1,):
            action = np.zeros(self.num_stocks + 1)
            action[0] = 1.0

        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum
        else:
            action[0] = 1.0

        if self.episode_steps % 100 == 0:
            self.debug_info['action'] = action.copy()

        current_prices = self.current_step['prices']
        portfolio_value = self.cash + np.sum(self.holdings * current_prices)

        target_values = action[1:] * portfolio_value
        current_values = self.holdings * current_prices
        delta_values = target_values - current_values

        transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

        for i in range(self.num_stocks):
            if current_prices[i] > 0:
                self.holdings[i] += delta_values[i] / current_prices[i]

        self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs

        old_portfolio_value = portfolio_value

        try:
            self.current_step = next(self.data_iterator)
            new_prices = self.current_step['prices']
            done = False
        except StopIteration:
            new_prices = current_prices
            done = True

        new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
        reward = (new_portfolio_value / old_portfolio_value) - 1
        reward -= transaction_costs / old_portfolio_value

        if self.episode_steps % 100 == 0 or done:
            self.debug_info.update({
                'portfolio_value_before': old_portfolio_value,
                'portfolio_value_after': new_portfolio_value,
                'transaction_costs': transaction_costs,
                'reward': reward,
                'cash': self.cash,
                'holdings_sum': np.sum(self.holdings * new_prices)
            })
            print(f"Step {self.episode_steps}: Reward = {reward:.6f}, "
                  f"Portfolio Value: {new_portfolio_value:.2f}, "
                  f"Transaction Costs: {transaction_costs:.2f}")

        self.episode_reward += float(reward)

        return (
            self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
            float(reward),
            done,
            False,
            {
                "portfolio_value": new_portfolio_value,
                "total_reward": self.episode_reward,
                "transaction_costs": transaction_costs,
                "cash": self.cash,
                "holdings_value": np.sum(self.holdings * new_prices)
            }
        )


### **FUNCTION TO RECOMMEND STOCKS**

In [29]:
import numpy as np
import pandas as pd
from stable_baselines3 import SAC

class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30):
        self.model = SAC.load(model_path)  # Load trained SAC model
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load recent stock market data
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent date for prediction
        self.latest_date = data['date'].max()

        # Select top stocks based on average trading volume
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        self.tickers = sorted(top_tickers)

        # Store latest stock prices
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Prepare technical indicators as input features
        self._prepare_features(data)

    def _prepare_features(self, data):
        """Compute market indicators for model input."""
        filtered_data = data[data['tic'].isin(self.tickers)]
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Feature matrix
        features = np.zeros((self.max_stocks, self.lookback, 9), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                returns = ticker_data['close'].pct_change().fillna(0).values
                volume_ma = ticker_data['volume'].rolling(window=10, min_periods=1).mean().values
                price_ma = ticker_data['close'].rolling(window=20, min_periods=1).mean().values

                delta = ticker_data['close'].diff().fillna(0)
                gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
                loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
                rs = gain / (loss + 1e-8)
                rsi = 100 - (100 / (1 + rs)).values

                exp1 = ticker_data['close'].ewm(span=12, adjust=False).mean()
                exp2 = ticker_data['close'].ewm(span=26, adjust=False).mean()
                macd = (exp1 - exp2).values

                volatility = ticker_data['close'].pct_change().rolling(window=30, min_periods=1).std().fillna(0).values
                momentum = ticker_data['close'].pct_change(periods=10).fillna(0).values

                bb_upper = price_ma + (volatility * 2)
                bb_lower = price_ma - (volatility * 2)

                for j in range(self.lookback):
                    features[i, j, 0] = returns[j]
                    features[i, j, 1] = volume_ma[j]
                    features[i, j, 2] = price_ma[j]
                    features[i, j, 3] = rsi[j]
                    features[i, j, 4] = macd[j]
                    features[i, j, 5] = volatility[j]
                    features[i, j, 6] = momentum[j]
                    features[i, j, 7] = bb_upper[j]
                    features[i, j, 8] = bb_lower[j]

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        """Recommend stock allocations using SAC model."""
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize actions to sum up to 1
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum
        else:
            action[0] = 1.0  # Default to all cash if allocation is invalid

        allocations = {}
        cash_allocation = action[0] * amount_cad

        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        print(f"\nAllocation Summary:")
        print(f"Cash: {action[0]*100:.2f}%")
        print(f"Stocks: {(1-action[0])*100:.2f}%")
        print(f"Number of stocks allocated: {len(allocations)}")

        if len(allocations) > 0:
            allocations_array = np.array([details['allocation_percent'] for details in allocations.values()])
            print(f"Max allocation: {np.max(allocations_array):.2f}%")
            print(f"Min allocation: {np.min(allocations_array):.2f}%")
            print(f"Avg allocation: {np.mean(allocations_array):.2f}%")

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }


### **EVALUATING THE MODEL**

In [30]:
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10):
    """Evaluate the SAC model on a validation dataset"""
    print(f"Evaluating SAC model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            # SAC model predict - action will be continuous (e.g., portfolio allocation)
            action, _ = model.predict(obs, deterministic=True)

            # Take step in the environment
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                      f"Steps = {step_count}, "
                      f"Final Value = ${final_value:.2f}, "
                      f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    print(f"\nEvaluation Results (over {episodes} episodes):")
    print(f"Average Total Reward: {avg_reward:.4f}")
    print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
    print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values


# **TRAINING THE SAC MODEL**

In [31]:
import gym
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from datetime import datetime

def train_model(data_path, model_save_path, max_stocks=100, lookback=30, timesteps=500000):
    print(f"Starting training at {datetime.now()}")

    # Initialize components with more manageable parameters
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20
    )

    def make_env():
        return PortfolioTradingEnv(train_iter)

    # Initialize SAC model
    model = SAC(
        "MlpPolicy",  # MLP (multi-layer perceptron) policy is typically used for stock trading
        DummyVecEnv([make_env]),  # Vectorized environment
        learning_rate=3e-4,
        buffer_size=50000,  # Replay buffer size
        batch_size=64,
        gamma=0.99,  # Discount factor
        tau=0.005,  # Soft update of target networks (SAC-specific)
        ent_coef="auto",  # Automatic entropy tuning
        verbose=1,
        device="auto",  # Use GPU if available
        tensorboard_log="./sac_portfolio_tensorboard/"  # Add tensorboard logging
    )

    # Log initial random policy performance
    print("Evaluating random policy before training...")
    try:
        obs = train_env.reset()
        done = np.array([False])
        total_reward = 0
        step_count = 0
        max_steps = 100  # Limit evaluation to avoid infinite loops

        while not done.any() and step_count < max_steps:
            actions = []
            for i in range(train_env.num_envs):
                # Generate an action vector of the correct shape (num_stocks + 1)
                action = np.random.random(train_iter.num_stocks + 1)
                # Normalize to sum to 1
                action = action / action.sum()
                actions.append(action)

            actions = np.array(actions)
            obs, rewards, done, info = train_env.step(actions)
            total_reward += rewards[0]  # Just take the first environment's reward
            step_count += 1

        print(f"Random policy evaluation: Total steps: {step_count}, Total reward: {total_reward}")
    except Exception as e:
        print(f"Error during random policy evaluation: {e}")
        print("Continuing with training anyway...")

    # Start training with progress monitoring
    print(f"Training model with {train_iter.num_stocks} stocks, each with {lookback} days of history")

    # Train in multiple stages with evaluation
    stage_timesteps = timesteps // 5
    for stage in range(5):
        print(f"\nTraining stage {stage+1}/5 ({stage_timesteps} timesteps)...")
        try:
            model.learn(total_timesteps=stage_timesteps, progress_bar=True, reset_num_timesteps=False)

            # Save checkpoint
            checkpoint_path = f"{model_save_path}_checkpoint_{stage+1}"
            model.save(checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Quick evaluation
            if stage < 4:  # Skip final evaluation as we'll do a full one later
                print("Quick evaluation of current policy...")
                eval_env = make_env()
                for i in range(3):  # Just a quick check with 3 episodes
                    obs, _ = eval_env.reset()
                    done = False
                    episode_reward = 0
                    step_count = 0
                    max_eval_steps = 100  # Limit evaluation steps

                    while not done and step_count < max_eval_steps:
                        action, _ = model.predict(obs, deterministic=True)
                        obs, reward, done, _, info = eval_env.step(action)
                        episode_reward += reward
                        step_count += 1

                    print(f"  Episode {i+1} reward: {episode_reward:.4f}, Steps: {step_count}, "
                          f"Final portfolio: ${info['portfolio_value']:.2f}")
        except Exception as e:
            print(f"Error during training stage {stage+1}: {e}")
            print("Attempting to continue with next stage...")

    # Save the final trained model
    try:
        model.save(model_save_path)
        print(f"Training completed at {datetime.now()} and model saved to {model_save_path}!")
    except Exception as e:
        print(f"Error saving model: {e}")
        print("Training completed but model could not be saved.")

    return model


### **PORTFOLIO METRICS**

In [32]:
def calculate_portfolio_metrics(portfolio_values, risk_free_rate=0.02):
    """Calculate common portfolio performance metrics"""
    if len(portfolio_values) < 2:
        return {
            "total_return": 0,
            "sharpe_ratio": 0,
            "max_drawdown": 0,
            "volatility": 0
        }

    # Calculate returns
    returns = np.array([(portfolio_values[i] / portfolio_values[i-1]) - 1
                       for i in range(1, len(portfolio_values))])

    # Calculate metrics
    total_return = (portfolio_values[-1] / portfolio_values[0]) - 1
    daily_returns_mean = np.mean(returns)
    daily_returns_std = np.std(returns)

    # Annualize (assuming 252 trading days)
    annualized_return = ((1 + daily_returns_mean) ** 252) - 1
    annualized_vol = daily_returns_std * np.sqrt(252)

    # Sharpe ratio
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else 0

    # Maximum drawdown
    peak = portfolio_values[0]
    max_drawdown = 0

    for value in portfolio_values:
        if value > peak:
            peak = value
        drawdown = (peak - value) / peak
        max_drawdown = max(max_drawdown, drawdown)

    return {
        "total_return": total_return * 100,  # Convert to percentage
        "annualized_return": annualized_return * 100,
        "volatility": annualized_vol * 100,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown * 100,
        "win_rate": np.mean(returns > 0) * 100
    }



# **BACKTEST**

In [33]:
def backtest_portfolio_sac(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=1):
    """Run a comprehensive backtest of the portfolio strategy with the SAC model"""
    print(f"Running {years}-year backtest simulation using SAC model...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        # Get action from the SAC model
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])  # Cash allocation is the first action value

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step in the environment
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress every 50 steps
        if step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics using the portfolio values
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results


#**MAIN FUNCTION**

In [ ]:
if __name__ == "__main__":
    import sys
    import os
    import json
    from datetime import datetime
    import numpy as np

    # Default parameters
    DATA_PATH = "/content/historical_data.csv"
    MODEL_SAVE_PATH = "portfolio_sac_model"
    MAX_STOCKS = 100  # Limit number of stocks to make training feasible
    LOOKBACK = 30     # Shorter lookback period
    TIMESTEPS = 500000  # Increased number of training steps
    INITIAL_CASH = 10000
    MODE = "train_and_evaluate"  # Default mode

    # Check if running in Jupyter/Colab environment
    is_notebook = 'ipykernel' in sys.modules

    if is_notebook:
        # For Jupyter/Colab, don't use argparse
        print("Running in notebook environment. Using default parameters.")
        print(f"DATA_PATH: {DATA_PATH}")
        print(f"MODEL_SAVE_PATH: {MODEL_SAVE_PATH}")
        print(f"MAX_STOCKS: {MAX_STOCKS}")
        print(f"LOOKBACK: {LOOKBACK}")
        print(f"TIMESTEPS: {TIMESTEPS}")
        print(f"INITIAL_CASH: {INITIAL_CASH}")
        print(f"MODE: {MODE}")

        # If you want to change parameters, do it here
    else:
        # For command line execution, use argparse
        import argparse
        parser = argparse.ArgumentParser(description="Portfolio Optimization with Soft Actor-Critic (SAC)")
        parser.add_argument("--data_path", type=str, default=DATA_PATH, help="Path to historical data CSV")
        parser.add_argument("--model_path", type=str, default=MODEL_SAVE_PATH, help="Path to save/load model")
        parser.add_argument("--max_stocks", type=int, default=MAX_STOCKS, help="Maximum number of stocks to consider")
        parser.add_argument("--lookback", type=int, default=LOOKBACK, help="Lookback window size")
        parser.add_argument("--timesteps", type=int, default=TIMESTEPS, help="Number of training timesteps")
        parser.add_argument("--initial_cash", type=float, default=INITIAL_CASH, help="Initial cash for portfolio")
        parser.add_argument("--mode", type=str, default=MODE,
                            choices=["train", "evaluate", "backtest", "recommend", "train_and_evaluate"],
                            help="Operation mode")

        args = parser.parse_args()

        # Update variables from command line args
        DATA_PATH = args.data_path
        MODEL_SAVE_PATH = args.model_path
        MAX_STOCKS = args.max_stocks
        LOOKBACK = args.lookback
        TIMESTEPS = args.timesteps
        INITIAL_CASH = args.initial_cash
        MODE = args.mode

    # Create results directory
    os.makedirs("results", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Main execution logic
    if MODE == "train" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nSTARTING TRAINING\n{'='*80}")
        model = train_model(
            DATA_PATH,
            MODEL_SAVE_PATH,
            MAX_STOCKS,
            LOOKBACK,
            TIMESTEPS
        )
        print(f"SAC Model saved to {MODEL_SAVE_PATH}")
    else:
        # Load existing SAC model
        try:
            from stable_baselines3 import SAC
            print(f"Loading SAC model from {MODEL_SAVE_PATH}")
            model = SAC.load(MODEL_SAVE_PATH)
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Please train a model first or provide a valid model path.")
            exit(1)

    if MODE == "evaluate" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nEVALUATING SAC MODEL\n{'='*80}")
        # Evaluate the model
        rewards, values = evaluate_model(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            episodes=10
        )

        # Save evaluation results
        eval_results = {
            "rewards": [float(r) for r in rewards],
            "final_values": [float(v) for v in values],
            "avg_reward": float(np.mean(rewards)),
            "avg_final_value": float(np.mean(values)),
            "max_reward": float(np.max(rewards)),
            "min_reward": float(np.min(rewards)),
            "success_rate": float(np.mean(np.array(rewards) > 0) * 100)
        }

        with open(f"results/evaluation_{timestamp}.json", "w") as f:
            json.dump(eval_results, f, indent=2)

        print(f"Evaluation results saved to results/evaluation_{timestamp}.json")

    if MODE == "backtest":
        print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
        # Run comprehensive backtest
        backtest_results = backtest_portfolio_sac(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            INITIAL_CASH,
            years=2  # Use 2 years of data for backtest
        )

        # Print backtest summary
        metrics = backtest_results["metrics"]
        print("\nBacktest Summary:")
        print(f"Total Return: {metrics['total_return']:.2f}%")
        print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
        print(f"Volatility: {metrics['volatility']:.2f}%")
        print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
        print(f"Win Rate: {metrics['win_rate']:.2f}%")
        print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
        print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
        print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

        # Save backtest results (excluding large arrays)
        backtest_summary = {
            "initial_cash": INITIAL_CASH,
            "final_value": float(backtest_results["final_value"]),
            "trading_days": backtest_results["steps"],
            "metrics": {k: float(v) for k, v in metrics.items()},
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
            "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
        }

        with open(f"results/backtest_{timestamp}.json", "w") as f:
            json.dump(backtest_summary, f, indent=2)

        print(f"Backtest summary saved to results/backtest_{timestamp}.json")

    if MODE == "recommend":
        print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
        # Generate portfolio recommendation
        recommender = PortfolioRecommender(
            MODEL_SAVE_PATH,
            DATA_PATH,
            max_stocks=MAX_STOCKS,
            lookback=LOOKBACK
        )

        recommendation = recommender.recommend_portfolio(INITIAL_CASH)

        # Print recommendation
        print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
        print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
        print("\nStock allocations:")

        # Sort allocations by percentage (largest first)
        sorted_allocations = sorted(
            recommendation['stock_allocations'].items(),
            key=lambda x: x[1]['allocation_percent'],
            reverse=True
        )

        for ticker, details in sorted_allocations:
            if details['allocation_percent'] > 1.0:  # Only show significant allocations
                print(f"{ticker}: ${details['allocation_cad']:.2f} "
                      f"({details['allocation_percent']:.2f}%) - "
                      f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

        # Calculate some portfolio statistics
        if sorted_allocations:
            allocations = [details['allocation_percent'] for _, details in sorted_allocations]
            print("\nPortfolio allocation statistics:")
            print(f"Number of stocks: {len(allocations)}")
            print(f"Max allocation: {max(allocations):.2f}%")
            print(f"Min allocation: {min(allocations):.2f}%")
            print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
            print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

        # Save recommendation
        with open(f"results/recommendation_{timestamp}.json", "w") as f:
            json.dump(recommendation, f, indent=2)

        print(f"Recommendation saved to results/recommendation_{timestamp}.json")

    print(f"\n{'='*80}\nPORTFOLIO OPTIMIZATION COMPLETE\n{'='*80}")
    print(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"All results saved in the 'results' directory.")


Running in notebook environment. Using default parameters.
DATA_PATH: /content/historical_data.csv
MODEL_SAVE_PATH: portfolio_sac_model
MAX_STOCKS: 100
LOOKBACK: 30
TIMESTEPS: 500000
INITIAL_CASH: 10000
MODE: train_and_evaluate

STARTING TRAINING
Starting training at 2025-03-24 19:49:27.298063
Loading data from /content/historical_data.csv...
Adding technical indicators...


Processing tickers: 100%|██████████| 100/100 [00:08<00:00, 11.98it/s]


Technical indicators added successfully
Scaling features...


Scaling features: 100%|██████████| 100/100 [00:08<00:00, 12.28it/s]


Preparing training data...


...

Step 2700: Reward = -0.014751, Portfolio Value: 7888.98, Transaction Costs: 2.89

Step 2800: Reward = -0.010185, Portfolio Value: 8130.35, Transaction Costs: 2.97

Step 2900: Reward = -0.007749, Portfolio Value: 9201.57, Transaction Costs: 2.84

Step 3000: Reward = 0.012713, Portfolio Value: 9857.04, Transaction Costs: 3.84

Step 3100: Reward = 0.001193, Portfolio Value: 8877.76, Transaction Costs: 3.60